# Dane do generacji

### Ilość danych

In [254]:
# People data
number_of_students = 5000
number_of_employees = 100
number_of_translators = 10

# Webinar data
number_of_webinars = 20
numbers_of_bought_webinars = 500

### Dane globalne 

In [255]:
# People
students = []
employees = []
translators = []

# Studies
meetings = set()
subjects = set()
field_of_studies = []

# students & field & dates
student_list = []

# Potentially low absence students
low_attendance_students_ID = set()

# Courses
courses = []
modules = []

### Zapewnienie unikalności

In [256]:
# Date & enployee ID
date_employee = set()
# Date & tramslator ID
date_translator = set()
# occupied rooms - room ID & date
room_and_hour = set()
# subject ID & date
subject_and_date = set()
# Students used in studies & field ID
used_students = set()
# Rooms used during session
used_rooms = set()
# Date & ModuleID
module_and_date = set()

# Generowanie danych do plików csv - pełen kod

### Użyte biblioteki

In [257]:
from faker import Faker
import random
from countryinfo import CountryInfo
import csv
import string
import unicodedata
from datetime import datetime, timedelta
import os
import math

### Funkcja używana do generowania adresów mailowych

In [258]:
def remove_special_characters(text):
    normalized_string = unicodedata.normalize('NFD', text)
    final_string = ''.join(
        char for char in normalized_string if ord(char) <= 127 and char.isalnum() and not char.isspace()
    )
    return final_string

### Zapisywanie danych do pliku .csv

In [259]:
def SavetoCsv(filename, data):
    
    folder = 'data'
    os.makedirs(folder, exist_ok=True) 
    filepath = os.path.join(folder, filename)
    
    with open(filepath, mode='w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        for row in data:
            writer.writerow(row)

# Generowanie informacji o ludziach z podziałem na studentów i pracowników

Generator gwarantuje unikatowość identyfikatotów, adresów email oraz numerów telefonów. Obecna implementacja umożliwia generowanie danych dla różnych krajów.

### Klasa Person

In [260]:
class Person:
    
    student_counter = 1000
    employee_counter = 1000
    translator_counter = 100
    generated_phone_numbers = set()
    generated_emails = set()
    
    def __init__(self, position, symbol):
        self.firstNameGenerate(symbol)
        self.lastNameGenerate(symbol)
        self.status = position
        self.birthDateGenerate()
        self.cityGenerate(symbol)
        self.streetAddressGenerate(symbol)
        self.getCountryName(symbol)
        self.getStudentID()
        self.getEmployeeID()
        self.getTranslatorID()
        self.phoneNumberGenerate(symbol)
        self.emailGenerate()
        self.translator_languages()
        self.jobPosition = None
        self.subjects = []
    
    def firstNameGenerate(self, symbol):
        fake = Faker(symbol)
        self.firstName = fake.first_name()
    
    def lastNameGenerate(self, symbol):
        fake = Faker(symbol)
        self.lastName = fake.last_name()
        
    def cityGenerate(self, symbol):
        fake = Faker(symbol)
        self.city = fake.city()
    
    def streetAddressGenerate(self, symbol):
        fake = Faker(symbol)
        self.streetAddress = fake.street_address()
    
    def getCountryName(self, symbol):
        
        country_dict = {
            "US": "Stany Zjednoczone",
            "PL": "Polska",
            "DE": "Niemcy",
            "GB": "Wielka Brytania",
            "FR": "Francja",
            "IT": "Włochy"
        }
        
        country_code = symbol.split('_')[1]
        self.country = country_dict[country_code]
    
    
    def birthDateGenerate(self):
        position = self.status
        fake = Faker()
        match position:
            case 'student':
                age_ranges = [(18, 25), (26, 30), (31, 50), (51, 65)]
                weights = [0.6, 0.2, 0.15, 0.05]
            case 'employee':
                age_ranges = [(25, 30), (31, 50), (51, 65), (66, 75)]
                weights = [0.2, 0.4, 0.3, 0.1]
            case 'translator':
                age_ranges = [(25, 30), (31, 50), (51, 65), (66, 75)]
                weights = [0.4, 0.3, 0.2, 0.1]
                
            
        selected_range = random.choices(age_ranges, weights=weights, k=1)[0]
        min_age, max_age = selected_range
        self.birthDate = fake.date_of_birth(None, min_age, max_age)
    
    def getStudentID(self):
        if self.status == 'student':
            self.studentID = Person.student_counter
            Person.student_counter += 1
    
    def getEmployeeID(self):
        if self.status == 'employee':
            self.employeeID = Person.employee_counter
            Person.employee_counter += 1
    
    def getTranslatorID(self):
        if self.status == 'translator':
            self.translatorID = Person.translator_counter
            Person.translator_counter += 1
    
    def phoneNumberGenerate(self, symbol):
        faker = Faker(symbol)
        country_code = symbol.split('_')[1]
        informations = CountryInfo(country_code)
        calling_code = informations.calling_codes()[0]
        
        while True:
            phone_number = faker.phone_number()
            if not phone_number[0] == '+':
                phone_number = "+" + calling_code + " " + phone_number
            if phone_number not in Person.generated_phone_numbers:
                self.phone = phone_number
                Person.generated_phone_numbers.add(phone_number)
                break
    
    def emailGenerate(self):
        domains = [
            "gmail.com",
            "outlook.com",
            "interia.pl",
            "yahoo.com",
            "wp.pl"
        ]
        weights = [0.4, 0.1, 0.2, 0.1, 0.2]
        first_name = remove_special_characters(self.firstName)
        last_name = remove_special_characters(self.lastName)
        
        while True:
            domain =  random.choices(domains, weights=weights, k=1)[0]
            random_number = random.randint(1, 9999)
            random_separator = random.choice([".", "_", "-"])
            
            email_prefix = random.choice([
            f"{first_name}{random_number}{last_name}",
            f"{first_name}{random_separator}{last_name}",
            f"{last_name}{random_separator}{first_name}",
            f"{last_name}{random_separator}{first_name}{random_number}",
            f"{first_name}{random_number}"
            f"{last_name}{random_separator}{random_number}"
            ]).lower()
            
            email = email_prefix + "@" + domain
            if email not in Person.generated_emails:
                self.email = email
                Person.generated_emails.add(email)
                break
            
    def translator_languages(self):
        if self.status == 'translator':
            languagesIDs = [1, 2, 3, 4, 5]
            numbers_of_languages = random.randint(1, 3)
            self.languagesID = random.sample(languagesIDs, k=numbers_of_languages)

### Przykład użycia

In [261]:

person = Person('employee', 'pl_PL')
print('First name:', person.firstName)
print('Last name:', person.lastName)
print('Position:', person.status)
print('Birth Date:', person.birthDate)
print('City:', person.city)
print('Address:', person.streetAddress)
print('Country:', person.country)
print('Employee ID:', person.employeeID)
print('Phone number:', person.phone)
print('Email:', person.email)


First name: Bruno
Last name: Dreszer
Position: employee
Birth Date: 1978-11-12
City: Chełm
Address: aleja Wojska Polskiego 65
Country: Polska
Employee ID: 1000
Phone number: +48 531 134 551
Email: dreszer.bruno@outlook.com


### Generowanie wszystkich studentów, pracowników i tłumaczy

In [206]:
for i in range(number_of_students):
    student = Person('student', 'pl_PL')
    students.append(student)
for i in range(number_of_employees):
    employee = Person('employee', 'pl_PL')
    employees.append(employee)
for i in range(number_of_translators):
    translator = Person('translator', 'pl_PL')
    translators.append(translator)

### Zapisywanie danych dla studentów do csv

In [207]:
students_info = []
for i in range(len(students)):
    student = students[i]
    student_data = [
        student.studentID, 
        student.firstName, 
        student.lastName, 
        student.birthDate, 
        student.country, 
        student.city, 
        student.streetAddress, 
        student.email, 
        student.phone
    ]
    students_info.append(student_data)
SavetoCsv('student.csv', students_info)

### Zapisywanie danych dla pracowników do csv

In [208]:
employees_info = []
for i in range(len(employees)):
    employee = employees[i]
    employee_data = [
        employee.employeeID, 
        employee.firstName, 
        employee.lastName, 
        employee.birthDate, 
        employee.country, 
        employee.city, 
        employee.streetAddress, 
        employee.email, 
        employee.phone
    ]
    employees_info.append(employee_data)
SavetoCsv('employee.csv', employees_info)

### Zapisywanie danych dla tłumaczy do pliku

In [209]:
translators_info = []
for i in range(len(translators)):
    translator = translators[i]
    translator_data = [
        translator.translatorID, 
        translator.firstName, 
        translator.lastName, 
        translator.birthDate, 
        translator.country, 
        translator.city, 
        translator.streetAddress, 
        translator.email, 
        translator.phone
    ]
    translators_info.append(translator_data)
SavetoCsv('translator.csv', translators_info)

# Generowanie języków

In [210]:
languages = {
    1 : 'Angielski',
    2 : 'Hiszpański',
    3 : 'Francuski',
    4 : 'Niemiecki',
    5 : 'Włoski',
    6 : 'Polski'
}

### Zapisywanie języków do csv

In [211]:
languages_info = []

for translator in translators:
    
    for language in translator.languagesID:
        languages_info.append([translator.translatorID, language])
        
SavetoCsv('translator_language.csv', languages_info)

languages_info = []

for i in range(1, 7):
    languages_info.append([i, languages[i]])
    
SavetoCsv('languages.csv', languages_info)

# Generowanie stanowisk pracowników

In [212]:
def generate_positions_to_csv():
    held_positions = []
    positions = [
        'Dyrektor główny', 
        'Księgowy', 
        'Księgowy',
        'Administrator danych',
        'Pracownik administracji'
    ]
    for position in positions:
        person = random.choice(employees)
        held_positions.append([person.employeeID, position])
        employees.remove(person)
    for person in employees:
        held_positions.append([person.employeeID, 'Prowadzący zajęcia'])
    SavetoCsv('positions.csv', held_positions)
generate_positions_to_csv()

# Generowanie informacji na temat lokalizacji zajęć

In [213]:
class LectureRoom:
    id_counter = 100
    lecture_rooms = set()
    
    def __init__(self):
        self.id = LectureRoom.id_counter
        self.buildingInfoGenerate()
        LectureRoom.id_counter += 1
    
    def buildingInfoGenerate(self):
        buildings = ['A-0', 'A-1', 'A-2', 'B-1', 'B-2', 'C-1', 'C-2', 'C-3']
        while True:
            floor = random.randint(1, 4)
            class_number = random.randint(1, 20)
            building = random.choice(buildings)
            new_class = str(class_number) + " " + str(floor) + building
            
            if new_class not in LectureRoom.lecture_rooms:
                self.building = building
                self.floor = floor
                self.classNumber = class_number
                LectureRoom.lecture_rooms.add(new_class)
                break

### Przykład generowanych sal lekcyjnych

In [214]:
classes = [LectureRoom(), LectureRoom(), LectureRoom()]

for i in range(3):
    print(f"\nSala {i}")
    print('Room ID:', classes[i].id)
    print('Building:', classes[i].building)
    print('Floor:', classes[i].floor)
    print('Class number:', classes[i].classNumber)


Sala 0
Room ID: 100
Building: A-2
Floor: 1
Class number: 3

Sala 1
Room ID: 101
Building: C-1
Floor: 3
Class number: 5

Sala 2
Room ID: 102
Building: C-1
Floor: 2
Class number: 7


### Generowanie danych dla sal i zapisanie ich

In [215]:
rooms_info = []
rooms = []
for i in range(30):
    room = LectureRoom()
    rooms.append(room)
    room_data = [
        room.id,
        room.building, 
        room.floor, 
        room.classNumber
    ]
    rooms_info.append(room_data)
SavetoCsv('lecturerooms.csv', rooms_info)

# Generowanie webinarów

W celu uniknięcia w przyszlości kolizji wynikającej z przypisania jednemu pracownikowi dwóch zajęć jednoczeńsnie, wszystkie daty wraz z ID pracownika będą zapisywane w jednym set.

### Nazwy webinarów

In [216]:
webinar_list = [
    "Wprowadzenie do programowania w Pythonie - podstawy i zaawansowane techniki",
    "Mistrzostwo w Data Science z Pythonem",
    "Budowanie skalowalnych aplikacji webowych z Django",
    "Uczenie maszynowe dla początkujących: przewodnik praktyczny",
    "Głębokie zanurzenie w struktury danych i algorytmy",
    "Technologia blockchain i jej zastosowania w IT",
    "Eksploracja chmurowych usług obliczeniowych z AWS",
    "Automatyzacja pracy z Pythonem i skryptami Bash",
    "Wprowadzenie do sztucznej inteligencji i uczenia głębokiego z TensorFlow",
    "Rozwój aplikacji webowych z JavaScript i React",
    "Zrozumienie DevOps: od rozwoju po wdrożenie",
    "Podstawy Kubernetes i Docker - jak zaczynać?",
    "Zabezpieczanie aplikacji webowych według najlepszych praktyk OWASP",
    "Tworzenie API z użyciem Flask i Pythona",
    "Budowanie interaktywnych dashboardów z Plotly i Dash",
    "Wprowadzenie do inżynierii danych i procesów ETL",
    "Eksploracja Internetu rzeczy (IoT) i obliczeń brzegowych",
    "Tworzenie bloga osobistego z Jekyll i GitHub Pages",
    "Projektowanie i tworzenie aplikacji mobilnych z React Native",
    "Budowanie modeli AI z Scikit-Learn",
    "Skuteczne zapytania SQL do analizy danych",
    "Budowanie aplikacji full-stack z Node.js i Express",
    "Wprowadzenie do chmurowych rozwiązań Microsoft Azure",
    "Programowanie funkcyjne w JavaScript",
    "Podstawy analizy danych z użyciem Pythona",
    "Zrozumienie architektury microservices w praktyce",
    "Tworzenie aplikacji webowych z Angular",
    "Optymalizacja wydajności aplikacji w Pythonie",
    "Wprowadzenie do UX/UI w projektowaniu aplikacji",
    "Analiza danych z użyciem R i Python",
    "Zarządzanie projektem IT z wykorzystaniem metod Agile",
    "Wprowadzenie do testowania oprogramowania automatycznego"
]

### Klasa webinar

In [217]:
class Webinar:
    
    webinar_counter = 1000
    generated_links = set()
    
    def __init__(self):
        self.getID()
        self.linkGenerate()
        self.priceGenerate()
        self.getEmployee()
        self.dateGenerate()
        self.getTranslator()
        self.videoLinkGenerate()
        self.getName()
        
        
    def getID(self):
        self.id = Webinar.webinar_counter
        Webinar.webinar_counter += 1
    
    def linkGenerate(self):
        prefix = "https://teams.microsoft.com/l/meetup-join/"
        while True:
            sufix = random.choices(string.ascii_lowercase + string.digits, k = 20)
            sufix = ''.join(sufix)
            if sufix not in Webinar.generated_links:
                link = prefix + sufix
                self.link = link
                Webinar.generated_links.add(link)
                break
    
    def priceGenerate(self):
        prices = [29.99, 99, 149, 249]
        weights = [0.3, 0.5, 0.15, 0.05]
        self.price = random.choices(prices, weights=weights, k=1)[0]
            
    
    def getEmployee(self):
        employee = random.choice(employees)
        self.employeeID = employee.employeeID
    
    def dateGenerate(self):
        teacher = self.employeeID
        hours = [
            '16:45',
            '18:30',
            '20:15'
        ]
        while True:
            hour = random.choice(hours)
            random_days = random.randint(-40, 90)    
            date = datetime.now() + timedelta(days=random_days)

            random_datetime_str = f"{date.strftime('%Y-%m-%d')} {hour}"
            random_datetime = datetime.strptime(random_datetime_str, '%Y-%m-%d %H:%M')
            
            if (random_datetime, teacher) not in date_employee:
                self.date = random_datetime
                date_employee.add((random_datetime, teacher))
                break
    
    def getTranslator(self):
        weights = [0.2, 0.8]
        while True:
            translator = random.choice(translators)
            translator = random.choices([translator, None], weights=weights, k=1)[0]
            if translator:
                languageID = random.choice(translator.languagesID)
            if translator is not None and languageID != 6:
                if (self.date, translator.translatorID) not in date_translator:
                    self.translatorID = translator.translatorID
                    self.languageID = languageID
                    date_translator.add((self.date, translator.translatorID))
                    break
                    
            else:
                self.translatorID = ''
                self.languageID = 6
                break
    
    def videoLinkGenerate(self):
        if self.date < datetime.now():
            prefix = "https://vimeo.com/"
            while True:
                sufix = random.choices(string.ascii_lowercase + string.digits, k = 40)
                sufix = ''.join(sufix)
                if sufix not in Webinar.generated_links:
                    link = prefix + sufix
                    self.videoLink = link
                    Webinar.generated_links.add(link)
                    break
        else:
            self.videoLink = ''
    
    def getName(self):
        name = random.choice(webinar_list)
        self.name = name
        webinar_list.remove(name)

### Przykładowy wygenerowany webinar

In [218]:
webinar = Webinar()
print('Webinar ID:', webinar.id)
print('Date and time:', webinar.date)
print('Employee ID:', webinar.employeeID)
print('Translator ID:', webinar.translatorID)
print('Language ID:', webinar.languageID)
print('Link:', webinar.link)
print('Video link:', webinar.videoLink)
print('Webinar name:', webinar.name)

Webinar ID: 1000
Date and time: 2024-12-06 16:45:00
Employee ID: 1035
Translator ID: 109
Language ID: 2
Link: https://teams.microsoft.com/l/meetup-join/rf7bj3nk24dxrp1jompk
Video link: https://vimeo.com/6vnepnywdwrg7bm4er5nr9acriivft0zsuove9h9
Webinar name: Podstawy analizy danych z użyciem Pythona


### Generowanie webinarów i zapisywanie ich do pliku

In [219]:
webinars = []
webinars_info = []
for i in range(20):
    webinar = Webinar()
    webinars.append(webinar)
    webinar_details = [
        webinar.id,
        webinar.price,
        webinar.date,
        webinar.languageID,
        webinar.translatorID,
        webinar.employeeID,
        webinar.link,
        webinar.videoLink,
        webinar.name
    ]
    webinars_info.append(webinar_details)
SavetoCsv('webinars.csv', webinars_info)

# Data ważności dostępu do webinaru oraz studenci korzystający z webinarów do csv

Ponieważ dostęp do platformy z dostępem jest przez miesiąc nie musimy przejmować się kolizją uczstnictwa studentów w webinarze z ich studiami

In [220]:
def webinar_expiration_date_to_csv(students_number):
    
    students_webinar = set()
    result_data = []
    
    for i in range(students_number):
        
        webinar = random.choice(webinars)
            
        while True:
            
            student = random.choice(students)
            
            if (student.studentID, webinar.id) not in students_webinar:
                
                students_webinar.add((student.studentID, webinar.id))
                break

        if webinar.date < datetime.now():
            
            expiration_date_start = webinar.date + timedelta(days=30)
            expiration_date_end = datetime.now() + timedelta(days=30)
            
            diff = (expiration_date_end - expiration_date_start).days
            expiration_date = (expiration_date_start + timedelta(days=random.randint(0, diff))).strftime('%Y-%m-%d')
            
        else:
            expiration_date = ''
            
        data = [webinar.id, student.studentID, expiration_date]
        result_data.append(data)
        
    SavetoCsv('webinar_expiration.csv', result_data)

webinar_expiration_date_to_csv(numbers_of_bought_webinars)   
            

# Generowanie danych dla studiów

### Kierunki studiów

In [221]:
class FieldOfStudies:
    
    field_id_counter = 1
    
    def __init__(self, name, description):
        self.getID()
        self.name = name
        self.limitGenerate()
        self.entryFeeGenerate()
        self.description = description
        
    def getID(self):
        self.fieldID = FieldOfStudies.field_id_counter
        FieldOfStudies.field_id_counter += 1
    
    def limitGenerate(self):
        lower_bound = 100
        difference = random.randint(0, 5)
        self.limit = lower_bound + difference * 10
    
    def entryFeeGenerate(self):
        self.fee = random.randint(9, 18) * 10

### Przedmioty

In [222]:
class Subject:
    
    subject_id_counter = 1
    
    def __init__(self, name, fieldID, description, semester):
        self.name = name
        self.fieldID = fieldID
        self.description = description
        self.getID()
        self.meetingQuantity()
        self.getEmployee()
        self.semester = semester
    
    def getID(self):
        self.id = Subject.subject_id_counter
        Subject.subject_id_counter += 1
    
    def meetingQuantity(self):
        self.quantity = random.randint(7, 18)
    
    #currently one employee could have multiple subjects and some of them have none of them
    def getEmployee(self):
        self.employeeID = random.choice(employees).employeeID

### Spotkania 

In [223]:
class Meeting:
    
    meeting_id_counter = 100
    
    def __init__(self, subjectID, type, date_range):
        self.getID()
        self.type = type
        self.linkGenerate()
        self.subjectID = subjectID
        self.dateGenerate(date_range)
        self.getTranslator()
        self.GetRoomID
        self.price = random.choice([29.99, 49.99, 59.99])
    
    def getID(self):
        self.id = Meeting.meeting_id_counter
        Meeting.meeting_id_counter += 1
        

    def dateGenerate(self, date_range):
        hours = [
            '10:00',
            '11:15',
            '13:00',
            '14:45'
        ]
        while True:
            hour = random.choice(hours)
            
            start_date_str = date_range[0]
            end_date_str = date_range[1]
            start_date = datetime.strptime(start_date_str, '%Y-%m-%d')
            end_date = datetime.strptime(end_date_str, '%Y-%m-%d')
            
            days_range = (end_date - start_date).days
            random_days = random.randint(0, days_range)
                
            date = start_date + timedelta(days=random_days)

            random_datetime_str = f"{date.strftime('%Y-%m-%d')} {hour}"
            random_datetime = datetime.strptime(random_datetime_str, '%Y-%m-%d %H:%M')
            
            if (random_datetime, self.subjectID) not in subject_and_date:
                self.date = random_datetime
                subject_and_date.add((random_datetime, self.subjectID))
                break
    
    def linkGenerate(self):
        if self.type == 2:
            prefix = "https://teams.microsoft.com/l/meetup-join/"
            while True:
                sufix = random.choices(string.ascii_lowercase + string.digits, k = 20)
                sufix = ''.join(sufix)
                if sufix not in Webinar.generated_links:
                    link = prefix + sufix
                    self.link = link
                    Webinar.generated_links.add(link)
                    break
        else:
            self.link = ''
    
    def GetRoomID(self):
        if self.type == 1:
            date = self.date
            while True:
                room = random.choice(rooms)
                if (room.id, date) not in room_and_hour:
                    self.roomID = room.id
                    room_and_hour.add((room.id, date))
                    break
        else:
            self.roomID = ''
    
    def getTranslator(self):
        weights = [0.2, 0.8]
        while True:
            
            translator = random.choice(translators)
            translator = random.choices([translator, None], weights=weights, k=1)[0]
            
            if translator:
                languageID = random.choice(translator.languagesID)
                
            if translator is not None and languageID != 6:
                
                if (self.date, translator.translatorID) not in date_translator:
                    
                    self.translatorID = translator.translatorID
                    self.languageID = languageID
                    date_translator.add((self.date, translator.translatorID))
                    break
            else:
                self.translatorID = ''
                self.languageID = 6
                break

# Generowanie danych dla studiów

### Zbiory oraz początkowe dane

In [224]:
it_programs = [
    {
        "name": "Informatyka ogólna",
        "description": "Kierunek obejmujący szeroką wiedzę z zakresu programowania, algorytmów oraz struktur danych.",
        "semesters": {
            1: [
                {"name": "Algorytmy i struktury danych", "description": "Nauka o algorytmach i strukturach danych, podstawy analizy algorytmów."},
                {"name": "Programowanie obiektowe", "description": "Zasady programowania w paradygmacie obiektowym, język Java lub C++."},
            ],
            2: [
                {"name": "Podstawy baz danych", "description": "Wprowadzenie do systemów baz danych, modelowanie danych, SQL."},
                {"name": "Systemy operacyjne", "description": "Podstawy działania systemów operacyjnych, zarządzanie pamięcią, procesami."},
            ],
            3: [
                {"name": "Sieci komputerowe", "description": "Wprowadzenie do sieci komputerowych, protokoły, topologie, bezpieczeństwo sieci."},
            ]
        }
    },
    {
        "name": "Inżynieria oprogramowania",
        "description": "Skupia się na tworzeniu, testowaniu i zarządzaniu projektami oprogramowania.",
        "semesters": {
            1: [
                {"name": "Projektowanie systemów informatycznych", "description": "Projektowanie i analiza systemów informatycznych, UML, diagramy."},
                {"name": "Testowanie oprogramowania", "description": "Metody testowania oprogramowania, testowanie jednostkowe, integracyjne, automatyczne."},
            ],
            2: [
                {"name": "Zarządzanie projektem IT", "description": "Metodyki zarządzania projektami IT, Agile, Scrum, Kanban."},
                {"name": "Bazy danych", "description": "Zaawansowane techniki pracy z bazami danych, normalizacja, transakcje."},
            ],
            3: [
                {"name": "Programowanie w języku Python", "description": "Podstawy programowania w języku Python, struktury danych, programowanie obiektowe."},
            ]
        }
    },
    {
        "name": "Sztuczna inteligencja",
        "description": "Kierunek związany z tworzeniem systemów inteligentnych, w tym rozwiązań z zakresu uczenia maszynowego.",
        "semesters": {
            1: [
                {"name": "Uczenie maszynowe", "description": "Metody uczenia maszynowego, algorytmy nadzorowane i nienadzorowane."},
                {"name": "Sieci neuronowe", "description": "Teoria i praktyka sieci neuronowych, ich zastosowanie w rozwiązywaniu problemów."},
            ],
            2: [
                {"name": "Analiza danych", "description": "Zbieranie, przetwarzanie i analiza danych przy użyciu narzędzi analitycznych."},
                {"name": "Algorytmy genetyczne", "description": "Wykorzystanie algorytmów inspirowanych ewolucją do rozwiązywania problemów optymalizacyjnych."},
            ],
            3: [
                {"name": "Rozpoznawanie obrazów", "description": "Przetwarzanie i analiza obrazów, rozpoznawanie wzorców przy użyciu sztucznej inteligencji."},
            ]
        }
    },
    {
        "name": "Cyberbezpieczeństwo",
        "description": "Skupia się na ochronie systemów komputerowych i sieci przed zagrożeniami z sieci.",
        "semesters": {
            1: [
                {"name": "Podstawy kryptografii", "description": "Teoria kryptografii, szyfrowanie, algorytmy kryptograficzne."},
                {"name": "Bezpieczeństwo systemów operacyjnych", "description": "Zabezpieczanie systemów operacyjnych przed atakami, zarządzanie uprawnieniami."},
            ],
            2: [
                {"name": "Analiza zagrożeń w sieciach komputerowych", "description": "Metody identyfikowania i eliminowania zagrożeń w sieciach komputerowych."},
                {"name": "Technologie ochrony danych", "description": "Zabezpieczanie danych przed nieautoryzowanym dostępem, zarządzanie prywatnością."},
            ],
            3: [
                {"name": "Ataki i obrona w cyberprzestrzeni", "description": "Techniki ataków i obrony przed nimi, analiza ataków DDoS, phishing."},
            ]
        }
    },
    {
        "name": "Big Data",
        "description": "Kierunek poświęcony analizie ogromnych zbiorów danych oraz wykorzystywaniu technologii do ich przetwarzania.",
        "semesters": {
            1: [
                {"name": "Przetwarzanie danych w chmurze", "description": "Wykorzystanie technologii chmurowych do przechowywania i przetwarzania dużych zbiorów danych."},
                {"name": "Hadoop i Spark", "description": "Platformy do przetwarzania dużych danych, Hadoop, Spark, MapReduce."},
            ],
            2: [
                {"name": "Analiza dużych zbiorów danych", "description": "Techniki analizy i przetwarzania dużych zbiorów danych, analiza statystyczna."},
                {"name": "Bazy danych NoSQL", "description": "Bazy danych NoSQL, ich struktura i zastosowania w analizie dużych danych."},
            ],
            3: [
                {"name": "Systemy rekomendacyjne", "description": "Tworzenie systemów rekomendacyjnych, algorytmy uczenia maszynowego w rekomendacjach."},
            ]
        }
    }
]


### Terminarz lat akademickich

In [225]:
time_ranges = {
    1: {
        "1 Semester": ('2023-10-01', '2024-01-20'),
        "1 Stationary Week": ('2024-01-21', '2024-01-28'),
        "2 Semester": ('2024-02-10', '2024-06-20'),
        "2 Stationary Week": ('2024-06-21', '2024-06-28')
    },
    2: {
        "1 Semester": ('2024-10-01', '2025-01-20'),
        "1 Stationary Week": ('2025-01-21', '2025-01-28'),
        "2 Semester": ('2025-02-10', '2025-06-20'),
        "2 Stationary Week": ('2025-06-21', '2025-06-28')
    }
}

### Generowanie kierunków i przedmiotów oraz spotkań

In [226]:
for program in it_programs:

    field = FieldOfStudies(program['name'], program['description'])
    field_of_studies.append(field)
    
    for semester, subjects_list in program['semesters'].items():
        for subject in subjects_list:
            new_subject = Subject(subject['name'], field.fieldID, subject['description'], semester)
            subjects.add(new_subject)
            
            if semester == 1 or semester == 3:
                for i in range(new_subject.quantity):
                    meeting = Meeting(new_subject.id, 2, ('2023-10-01', '2024-01-20'))
                    meetings.add(meeting)
                    meeting = Meeting(new_subject.id, 2, ('2024-10-01', '2025-01-20'))
                    meetings.add(meeting)
            if semester == 2:
                for i in range(new_subject.quantity):
                    meeting = Meeting(new_subject.id, 2, ('2024-02-10', '2024-06-20'))
                    meetings.add(meeting)

### Wizualizacja wygenerowanych przedmiotów

In [227]:
for subject in subjects:
    print(f'ID: {subject.id}')
    print(f'Field ID: {subject.fieldID}')
    print(f'Name: {subject.name}')
    print(f'Description: {subject.description}')
    print(f'Quantity: {subject.quantity}')
    print(f'Employee ID: {subject.employeeID}')
    print(f'Semester: {subject.semester}')
    print('-' * 50)

ID: 16
Field ID: 4
Name: Podstawy kryptografii
Description: Teoria kryptografii, szyfrowanie, algorytmy kryptograficzne.
Quantity: 14
Employee ID: 1028
Semester: 1
--------------------------------------------------
ID: 21
Field ID: 5
Name: Przetwarzanie danych w chmurze
Description: Wykorzystanie technologii chmurowych do przechowywania i przetwarzania dużych zbiorów danych.
Quantity: 12
Employee ID: 1018
Semester: 1
--------------------------------------------------
ID: 25
Field ID: 5
Name: Systemy rekomendacyjne
Description: Tworzenie systemów rekomendacyjnych, algorytmy uczenia maszynowego w rekomendacjach.
Quantity: 13
Employee ID: 1049
Semester: 3
--------------------------------------------------
ID: 1
Field ID: 1
Name: Algorytmy i struktury danych
Description: Nauka o algorytmach i strukturach danych, podstawy analizy algorytmów.
Quantity: 11
Employee ID: 1071
Semester: 1
--------------------------------------------------
ID: 22
Field ID: 5
Name: Hadoop i Spark
Description: Plat

### Zapisywanie do pliku csv

In [228]:
subjects_info = []
field_of_studies_info = []
meetings_info = []

for meeting in meetings:
    meetings_info.append([meeting.id, meeting.type, meeting.subjectID, meeting.date, meeting.link, meeting.languageID, meeting.translatorID, meeting.price])
SavetoCsv('Meeting.csv', meetings_info)

for subject in subjects:
    subjects_info.append([subject.id, subject.fieldID, subject.name, subject.description, subject.quantity, subject.employeeID, subject.semester])
SavetoCsv('Subjects.csv', subjects_info)

for field in field_of_studies:
    field_of_studies_info.append([field.fieldID, field.name, field.description, field.limit, field.fee])
SavetoCsv('Fields.csv', field_of_studies_info)

# Tabela typów spotkań

In [229]:
SavetoCsv('Meetingtype.csv', [[1, 'zajęcia w formie stacjonarnej'], [2, 'zajęcia w formie zdalnej']])

# Lista studentów dla każdego wydziału oraz rocznika

In [230]:
def students_list():

    for _, field in enumerate(field_of_studies):
        capacity_per_semester = field.limit
        
        # Generate students for third semester in '2023-10-01', '2024-01-20'
        diff = random.randint(30, 50)
        allocated_students = 0
        while allocated_students < capacity_per_semester - diff:
            student = random.choice(students)
            if (student, field.fieldID) not in used_students:
                used_students.add((student, field.fieldID))
                student_list.append([field.fieldID, student.studentID, 3, '2023-10-01', '2024-01-20'])
                allocated_students += 1

        # Generate students for first in '2023-10-01', '2024-01-20'
        diff = random.randint(50, 60)
        allocated_students = 0
        while allocated_students < capacity_per_semester - diff:
            student = random.choice(students)
            if (student, field.fieldID) not in used_students:
                used_students.add((student, field.fieldID))
                student_list.append([field.fieldID, student.studentID, 1, '2023-10-01', '2024-01-20'])
                student_list.append([field.fieldID, student.studentID, 2, '2024-02-10', '2024-06-20'])
                student_list.append([field.fieldID, student.studentID, 3, '2024-10-01', ''])
                allocated_students += 1
        
        # Generate students for first semester in '2024-10-01'
        diff = random.randint(41, 50)
        allocated_students = 0
        while allocated_students < capacity_per_semester - diff:
            student = random.choice(students)
            if (student, field.fieldID) not in used_students:
                used_students.add((student, field.fieldID))
                student_list.append([field.fieldID, student.studentID, 1, '2024-10-01', ''])
                allocated_students += 1

    SavetoCsv('FacultyStudentList.csv', student_list)

students_list()

# Oceny za przedmioty

In [231]:
low_attendance_students_ID = set()

def grades():
    grade_list = []
    weights = [0.3, 0.1, 0.25, 0.15, 0.1]  # Weights for grade distribution
    grades = [2, 3, 4, 5]  # Grades from 2 to 5

    for record in student_list:
        fieldID, studentID, semester, start_date, end_date = record
        
        # Check if the subject is completed
        if start_date and end_date:
            # Find all subjects associated with this field and semester
            for subject in subjects:
                if subject.fieldID == fieldID and subject.semester == semester:
                    # Assign a random grade
                    grade = random.choices(grades, weights=weights, k=1)[0]
                    if grade == 2:
                        low_attendance_students_ID.add((studentID, subject.id, semester, start_date, end_date))
                    grade_list.append([subject.id, studentID, grade])

    SavetoCsv('GradeList.csv', grade_list)

# Nieobecności na zajęciach

In [232]:
# Find all meetings in chosen date range
def find_meeting(subject_id, start_date, end_date):
    """
    Finds all dates for meetings with the given subject ID in the CSV file.
    """
    meeting_list = []
    
    start_date = datetime.strptime(start_date, '%Y-%m-%d')
    if end_date:
        end_date = datetime.strptime(end_date, '%Y-%m-%d')
    else:
        end_date = datetime.now()

    for meeting in meetings:
            
            if meeting.subjectID == subject_id and start_date <= meeting.date and end_date >= meeting.date:
                meeting_list.append(meeting)

    return meeting_list

# Find all subjects in certain semester and field
def find_subjects(fieldID, semester):
    
    subject_list = []
    
    for subject in subjects:
        
        if subject.fieldID == fieldID and subject.semester == semester:
            subject_list.append(subject)
    return subject_list
        
        
# Find all semesters and field for chosen student
def find_students_semesters_and_field(studentID):
    
    student_data = []
    
    for row in student_list:
        
        if row[1] == studentID:
            student_data.append((row[0], row[2], row[3], row[4]))
    
    return student_data

def find_extra_class(meeting):
    
    for next_meeting in meetings:
        
        if next_meeting.date > meeting.date:
            return next_meeting
    return ''
    

def absences():
    absence_info = []
    weights = [0.2, 0.2, 0.4, 0.2]
    absence_count = [0, 1, 2, 3]
    used_meetings = set()
    
    for student, field in used_students:
        
        for fieldID, semester, start_date, end_date in find_students_semesters_and_field(student.studentID):
            
            for subject in find_subjects(fieldID, semester):
                
                #number of absences for each subject
                number_of_abseces = random.choices(absence_count, weights=weights, k=1)[0]
            
                #find all meetings of this subject
                meeting_list = find_meeting(subject.id, start_date, end_date)
                
                # Number of meetings per semester
                quantity = subject.quantity
                required_quantity = math.ceil(subject.quantity *0.8)
            
                chosen_meetings = random.choices(meeting_list, k=number_of_abseces)

                # Situation when student doeas not have to go to extra classes
                if quantity - number_of_abseces >= required_quantity:
                    
                    for item in chosen_meetings:
                        if (item.id, student.studentID) not in used_meetings:
                            absence_info.append([student.studentID, item.id, ''])
                            used_meetings.add((item.id, student.studentID))  
                
                else:
                    number_of_make_up_classes = number_of_abseces - (quantity - required_quantity)
                    for item in chosen_meetings:
                        if (item, student.studentID) not in used_meetings and number_of_make_up_classes:
                            extra_class = find_extra_class(item)
                            absence_info.append([student.studentID, item.id, extra_class.id])
                            used_meetings.add((item.id, student.studentID))  
                            number_of_make_up_classes -= 1
                            
                        elif (item, student.studentID) not in used_meetings and number_of_make_up_classes == 0:
                            absence_info.append([student.studentID, item.id, ''])
                            used_meetings.add((item.id, student.studentID))  
                          
                

    SavetoCsv('StudentAbsence.csv', absence_info)
            
absences()

# Zjazdy na studiach

In [233]:
session_week = []

for field in field_of_studies:
    while True:
        room = random.choice(rooms)
        if room not in used_rooms:
            session_week.append([field.fieldID, 1, '2024-01-21', '2024-01-28'])
            used_rooms.add(room)
            break
    
    while True:
        room = random.choice(rooms)
        if room not in used_rooms:
            session_week.append([field.fieldID, 3, '2024-01-21', '2024-01-28'])
            used_rooms.add(room)
            break
    
    while True:
        room = random.choice(rooms)
        if room not in used_rooms:
            session_week.append([field.fieldID, 2, '2024-06-21', '2024-06-28'])
            used_rooms.add(room)
            break
    
    while True:
        room = random.choice(rooms)
        if room not in used_rooms:
            session_week.append([field.fieldID, 3, '2025-01-21', '2025-01-28'])
            used_rooms.add(room)
            break 
    
    while True:
        room = random.choice(rooms)
        if room not in used_rooms:
            session_week.append([field.fieldID, 1, '2025-01-21', '2025-01-28'])
            used_rooms.add(room)
            break 

# Praktyki na studiach

In [234]:
class Intership:
    
    id_count = 1
    
    def __init__(self, fieldID, semester, internships, start_date):
        self.internship(internships)
        self.semester = semester
        self.date(start_date)
        self.fieldID = fieldID
        self.setID()
        
    def setID(self):
        self.id = Intership.id_count
        Intership.id_count += 1
    
    def internship(self, internships):
        company = random.choice(internships)
        self.company = company
    
    def date(self, start_date):
        random_days = random.randint(0, 40)    
        start_date = datetime.strptime(start_date, '%Y-%m-%d')
        
        date = start_date + timedelta(days=random_days)
        self.start_date = date.strftime('%Y-%m-%d')
        self.end_date = (date + timedelta(days=14)).strftime('%Y-%m-%d')
        

In [235]:
companies_internships = [
    "Digital Equipment Corporation - Data Analyst Intern",
    "Compaq - Software Engineer Intern",
    "IBM - Systems Engineer Intern",
    "Sun Microsystems - Network Engineer Intern",
    "Atari - Game Developer Intern",
    "Apple Computer - UI/UX Designer Intern",
    "Microsoft - Software Development Intern",
    "Borland - QA Engineer Intern",
    "Novell - IT Support Intern",
    "SGI - Graphics Engineer Intern",
    "Tandy - Hardware Engineer Intern",
    "NeXT - Product Manager Intern",
    "Hewlett-Packard - Business Analyst Intern",
    "Xerox PARC - Research Intern",
    "Wang Laboratories - Data Scientist Intern",
    "Packard Bell - Marketing Intern",
    "Gateway Computers - Web Developer Intern",
    "Synapse Software - Mobile Developer Intern",
    "3Com - Network Security Intern",
    "Acer - IT Project Manager Intern",
    "Oracle - Database Administrator Intern",
    "Intel - Embedded Systems Intern",
    "Cisco - Cloud Engineer Intern",
    "NVIDIA - Machine Learning Intern",
    "Dell - Cybersecurity Analyst Intern",
    "Adobe - Front-End Developer Intern",
    "Google - AI Research Intern",
    "Yahoo - SEO Specialist Intern",
    "Tesla - Automation Engineer Intern",
    "SpaceX - Aerospace Software Intern"
]

In [236]:
interships = set()
def create_interships():
    used_companies = set()
    interships_info = []
    for field in field_of_studies:
        fieldID = field.fieldID

        while True:
            new_intership = Intership(fieldID, 1, companies_internships, '2023-10-01')
            if new_intership.company not in used_companies:
                interships.add(new_intership)
                used_companies.add(new_intership.company)
                interships_info.append([new_intership.id, new_intership.fieldID, new_intership.semester, new_intership.company, new_intership.start_date, new_intership.end_date])
                break
        
        while True:
            new_intership = Intership(fieldID, 3, companies_internships, '2023-10-01')
            if new_intership.company not in used_companies:
                interships.add(new_intership)
                used_companies.add(new_intership.company)
                interships_info.append([new_intership.id, new_intership.fieldID, new_intership.semester, new_intership.company, new_intership.start_date, new_intership.end_date])
                break
        
        while True:
            new_intership = Intership(fieldID, 1, companies_internships, '2024-10-01')
            if new_intership.company not in used_companies:
                interships.add(new_intership)
                used_companies.add(new_intership.company)
                interships_info.append([new_intership.id, new_intership.fieldID, new_intership.semester, new_intership.company, new_intership.start_date, new_intership.end_date])
                break
        
        while True:
            new_intership = Intership(fieldID, 3, companies_internships, '2024-10-01')
            if new_intership.company not in used_companies:
                interships.add(new_intership)
                used_companies.add(new_intership.company)
                interships_info.append([new_intership.id, new_intership.fieldID, new_intership.semester, new_intership.company, new_intership.start_date, new_intership.end_date])
                break
        
        while True:
            new_intership = Intership(fieldID, 2, companies_internships, '2024-02-10')
            if new_intership.company not in used_companies:
                interships.add(new_intership)
                used_companies.add(new_intership.company)
                interships_info.append([new_intership.id, new_intership.fieldID, new_intership.semester, new_intership.company, new_intership.start_date, new_intership.end_date])
                break
   
    SavetoCsv('Internships.csv', interships_info)

create_interships()

# Nieobecności na praktykach

In [237]:
# Find all semesters and field for chosen student
def find_students_semesters_and_field(studentID):
    
    student_data = []
    
    for row in student_list:
        
        if row[1] == studentID:
            student_data.append((row[0], row[2], row[3], row[4]))
    
    return student_data


def get_internships_by_fieldID(fieldID, semester, start_date, end_date):
    return [internship for internship in interships if (internship.fieldID == fieldID and internship.semester == semester and datetime.strptime(internship.start_date, '%Y-%m-%d') >= start_date and datetime.strptime(internship.start_date, '%Y-%m-%d') < end_date)][0]

def internship_absences():
    interships_absence_info = []
    weights = [0.9, 0.1]
    absence_count = [0, 1]
    
    for student, field in used_students:
        
        student_data = find_students_semesters_and_field(student.studentID)

        for fieldID, semester, start_date, end_date in student_data:
            
            start_date = datetime.strptime(start_date, '%Y-%m-%d')
            
            if end_date:
                end_date = datetime.strptime(end_date, '%Y-%m-%d')
            else:
                end_date = datetime.now()
            
            internship = get_internships_by_fieldID(fieldID, semester, start_date, end_date)
            
            number = random.choices(absence_count, weights=weights, k=1)[0]
            
            if number != 0:
                
                date = (datetime.strptime(internship.start_date, '%Y-%m-%d') + timedelta(days=random.randint(0, 13))).strftime('%Y-%m-%d') 
                interships_absence_info.append([internship.id, student.studentID, date])
            
             
    
    SavetoCsv('InternshipAbsences.csv', interships_absence_info)          

internship_absences()

# Generowanie kursów

### Klasa kursów 

In [249]:
course_list = [
    "Web Development with HTML and CSS",
    "Introduction to JavaScript",
    "Advanced React.js",
    "Backend Development with Node.js",
    "Database Management with SQL"
]

In [239]:
class Course:
    
    id_counter = 1
    
    def __init__(self, names):
        self.getID()
        self.getPrice()
        self.getType()
        self.getEmployee()
        self.limitGenerate()
        self.getTranslator()
        self.modules_quantity = random.randint(3, 5)
        self.getName(names)
        
    def getID(self):
        self.id = Course.id_counter
        Course.id_counter += 1
        
    def getPrice(self):
        weights = [0.2, 0.4, 0.2, 0.1, 0.1]
        prices = [599, 999, 1499, 2999, 4999]
        price = random.choices(prices, weights=weights, k=1)[0]
        self.price = price
    
    def getType(self):
        typeMeetings = ['zdalny', 'stacjonarny', 'asynchroniczny']
        weights = [0.7, 0.2, 0.1]
        courseType = random.choices(typeMeetings, weights=weights, k=1)[0]
        self.type = courseType
    
    def getEmployee(self):
        employee = random.choice(employees)
        self.employeeID = employee.employeeID
    
    def limitGenerate(self):
        if self.type != 'zdalny':
            lower_bound = 100
            difference = random.randint(0, 5)
            self.limit = lower_bound + difference * 10
        else:
            self.limit =''
    
    def getTranslator(self):
        weights = [0.2, 0.8]
        translator = random.choice(translators)
        translator = random.choices([translator, None], weights=weights, k=1)[0]
        if translator:
            languageID = random.choice(translator.languagesID)
        if translator is not None and languageID != 6:
            self.translatorID = translator.translatorID
            self.languageID = languageID
        else:
            self.translatorID = ''
            self.languageID = 6
    
    def getName(self, names):
        name = random.choice(names)
        self.name = name
        names.remove(name)

### Klasa module

In [250]:
module_list = [
    "Wprowadzenie do kursu i celów nauki",
    "Fundamenty teoretyczne i podstawowe pojęcia",
    "Przegląd narzędzi i technologii używanych w kursie",
    "Instalacja i konfiguracja środowiska pracy",
    "Podstawowe operacje i pierwsze kroki",
    "Zrozumienie kluczowych koncepcji i zastosowań",
    "Praktyczne ćwiczenia na podstawie rzeczywistych przykładów",
    "Praca z danymi wejściowymi i wyjściowymi",
    "Projektowanie i implementacja rozwiązań",
    "Analiza problemów i strategie ich rozwiązywania",
    "Debugowanie, testowanie i poprawa jakości kodu",
    "Najlepsze praktyki w projektowaniu i wdrażaniu",
    "Zaawansowane techniki i optymalizacja procesów",
    "Integracja zewnętrznych bibliotek i narzędzi",
    "Implementacja architektury projektu",
    "Tworzenie dynamicznych komponentów i elementów",
    "Zarządzanie wydajnością i skalowalnością",
    "Automatyzacja i zarządzanie procesami",
    "Przygotowanie dokumentacji technicznej",
    "Praktyczne case studies i rozwiązania problemów",
    "Zarządzanie projektami i wersjonowanie kodu",
    "Podsumowanie kluczowych umiejętności",
    "Planowanie dalszego rozwoju i nauki",
    "Egzamin końcowy weryfikujący wiedzę",
    "Realizacja projektu końcowego i prezentacja wyników"
]

In [241]:
class Module:
    
    id_counter = 1
    
    def __init__(self, course):
        self.getID()
        self.courseID = course.id
        self.getType(course)
        self.quantity = random.randint(2, 5)
        self.getName()
        
    def getID(self):
        self.id = Module.id_counter
        Module.id_counter += 1
    
    def getType(self, course):
        type = course.type
        if type != 'asynchroniczny':
            self.type = type
        else:
            self.type = random.choice(['zdalny', 'stacjonarny'])
    
    def getName(self):
        name = random.choice(module_list)
        self.name = name
        module_list.remove(name)

### Tworzenie modułów i kursów

In [251]:
def generate_courses(course_quantity):
    
    courses_info = []
    modules_info = []
    
    for _ in range(course_quantity):
        
        course = Course(course_list)
        courses.append(course)
        courses_info.append([course.id, course.employeeID, course.price, course.type, course.limit, course.languageID, course.translatorID, course.modules_quantity, course.name])
        
        for _ in range(course.modules_quantity):
            
            module = Module(course)
            modules.append(module)    
            modules_info.append([module.id, module.courseID, module.type, module.quantity, module.name])

    SavetoCsv('Courses.csv', courses_info)
    SavetoCsv('Modules.csv', modules_info)
generate_courses(5)

### Przykładowy kurs

In [252]:
course = random.choice(courses)
print('Course ID:', course.id)
print('Employee ID:', course.employeeID)
print('Price:', course.price)
print('Course type:', course.type)
print('Course limit:', course.limit)
print('Language ID:', course.languageID)
print('Translator ID:', course.translatorID)
print('Modules quantity:', course.modules_quantity)
print('Course Name:', course.name)

Course ID: 9
Employee ID: 1024
Price: 4999
Course type: zdalny
Course limit: 
Language ID: 6
Translator ID: 
Modules quantity: 5
Course Name: Database Management with SQL


### Przykładowy moduł

In [253]:
module = random.choice(modules)
print('Module ID:', module.id)
print('Course ID:', module.courseID)
print('Meetings quantity:', module.quantity)
print('Module Name:', module.name)

Module ID: 34
Course ID: 12
Meetings quantity: 2
Module Name: Egzamin końcowy weryfikujący wiedzę


# Harmonogram studiów

In [ ]:
class CourseMeeting:
    
    id_counter = 1
    
    def __init__(self, module):
        self.moduleID = module.id
        self.dateGenerate()
        self.type = module.type
        self.GetRoomID()
        self.videoLinkGenerate()
        self.getID()
        
    def getID(self):
        self.id = CourseMeeting.id_counter
        CourseMeeting.id_counter += 1
    
    def dateGenerate(self):
        hours = [
            '10:00',
            '11:15',
            '13:00',
            '14:45'
        ]
        while True:
            hour = random.choice(hours)
                
            date = datetime.now() + timedelta(days=random.randint(-100, 20))

            random_datetime_str = f"{date.strftime('%Y-%m-%d')} {hour}"
            random_datetime = datetime.strptime(random_datetime_str, '%Y-%m-%d %H:%M')
            
            if (random_datetime, self.moduleID) not in module_and_date:
                self.date = random_datetime
                module_and_date.add((random_datetime, self.moduleID))
                break
    
    def linkGenerate(self):
        if self.type == 'zdalny':
            prefix = "https://teams.microsoft.com/l/meetup-join/"
            while True:
                sufix = random.choices(string.ascii_lowercase + string.digits, k = 20)
                sufix = ''.join(sufix)
                if sufix not in Webinar.generated_links:
                    link = prefix + sufix
                    self.link = link
                    Webinar.generated_links.add(link)
                    break
        else:
            self.link = ''
    
    def GetRoomID(self):
        if self.type == 'stacjonarny':
            date = self.date
            while True:
                room = random.choice(rooms)
                if (room.id, date) not in room_and_hour:
                    self.roomID = room.id
                    room_and_hour.add((room.id, date))
                    break
        else:
            self.roomID = ''
    
    def videoLinkGenerate(self):
        if self.date < datetime.now():
            prefix = "https://vimeo.com/"
            while True:
                sufix = random.choices(string.ascii_lowercase + string.digits, k = 40)
                sufix = ''.join(sufix)
                if sufix not in Webinar.generated_links:
                    link = prefix + sufix
                    self.videoLink = link
                    Webinar.generated_links.add(link)
                    break
        else:
            self.videoLink = ''

### Generowanie harmonogramu

In [ ]:
course_schedule = []
course_meetings = set()

for module in modules:
    
    for _ in range(module.quantity):
        
        meeting = CourseMeeting(module)
        course_meetings.add(meeting)
        course_schedule.append([meeting.id, meeting.roomID, meeting.link, meeting.videoLink, meeting.date])

SavetoCsv('CourseSchedule.csv', course_schedule)